In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

115

In [3]:
documents = documents_llm

## Generate questions

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
import json

user_prompt = json.dumps(doc)

In [8]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [9]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [10]:
result = response.output_parsed

from pprint import pprint
pprint(result.questions)

["Why doesn't my homework result match any of the multiple-choice answers, "
 'even though my code seems fine?',
 "Could the issue be that I'm slicing columns or filtering in the wrong order "
 'before using head() or values?',
 "Is it possible I applied log transformation at the wrong step and that's why "
 'my answer is off?',
 "Can rounding too early change the final homework answer enough that it won't "
 'match the choices?',
 'Could different sklearn, numpy, or split logic versions make my output '
 'different from the expected one?']


In [11]:
from evaluation_utils import llm_structured, calc_price

/workspaces/llm-zoomcamp-2026/week_4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

["Why doesn't my homework result match any of the multiple-choice options?", 'What are the most common reasons my answer is off when I compare it to the listed choices?', 'Could the issue be that I sliced the columns or filtered the data in the wrong order?', 'How much does early rounding or a log transform affect the final homework answer?', "If I still can't get an exact match after checking everything, am I allowed to choose the closest option?"]


In [13]:
print(calc_price(usage))

{'input_cost': 0.000264, 'output_cost': 0.000459, 'total_cost': 0.000723}


In [14]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': "Why doesn't my homework result match any of the multiple-choice options?",
  'document': 'ab183bd688'},
 {'question': 'What are the most common reasons my answer is off when I compare it to the listed choices?',
  'document': 'ab183bd688'},
 {'question': 'Could the issue be that I sliced the columns or filtered the data in the wrong order?',
  'document': 'ab183bd688'},
 {'question': 'How much does early rounding or a log transform affect the final homework answer?',
  'document': 'ab183bd688'},
 {'question': "If I still can't get an exact match after checking everything, am I allowed to choose the closest option?",
  'document': 'ab183bd688'}]

In [15]:
import pandas as pd

In [16]:
pd.DataFrame(records)

,question,document
0,Why doesn't my homework result match any of th...,ab183bd688
1,What are the most common reasons my answer is ...,ab183bd688
2,Could the issue be that I sliced the columns o...,ab183bd688
3,How much does early rounding or a log transfor...,ab183bd688
4,If I still can't get an exact match after chec...,ab183bd688


In [17]:
from evaluation_utils import llm_structured_retry

In [18]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [19]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:08<00:00,  1.76s/it]


In [20]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [21]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

100%|██████████| 115/115 [00:33<00:00,  3.38it/s]


In [22]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

575

In [28]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08843025000000003

In [29]:
df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)